# Transport boxes of item BT9 on Google Colab

This notebook runs boxes of molecular dynamics (MD) for plan item BT9 of the CMF project. Each
box is one electrolyte: LiPF6 in a mixture of carbonate solvents, 1,000 solvent molecules, 4 ns
of production at 298 K, with OpenMM on the graphics card. The code is the code of
`benchmarks/run_diffusion.py`, and the settings are the same as on the author's desktop.

**What you need first**

1. A Colab runtime with a GPU: *Runtime, Change runtime type, GPU*. A T4 is enough.
2. The file `cmf_colab_bundle.zip` in a folder `cmf_colab` in *My Drive*. Make it on your own
   computer with `python benchmarks/colab/make_bundle.py`. Nothing is uploaded by the script.

**How to use it**

1. Change the settings in the first code cell if you want to. The default runs the 40 other
   formulations (`RUNS = "rest"`) and starts no box that would not finish within 10 hours. On a
   T4 one box takes about 20 minutes and 0.4 compute units. For a timing run of one box, set
   `MAX_BOXES = 1`.
2. Run the cells from the top, one after the other. The install cell takes several minutes.
3. Run the notebook again in a new session if boxes are left. Boxes that are already in Drive
   are skipped. The runner takes the list from the end, so a run on your own computer can take
   it from the front.
4. Copy the box folders from `My Drive/cmf_colab/transport` into
   `benchmarks/cache/transport` on your own computer. Then run
   `python benchmarks/run_diffusion.py --stage analyse`.

**What it costs.** Compute units are used while the runtime is connected, also when it is idle.
The last cell ends the runtime when the run stops. A notebook runs at most 12 hours. Unless your
plan has background execution, keep the browser tab open while a box runs.

In [ ]:
# --- Settings. Change them here, then run all cells from the top. ---
DRIVE_FOLDER = "cmf_colab"      # a folder in My Drive: it holds the bundle and gets the results
RUNS = "rest"                   # "rest" = the 40 other formulations, "transfer" = the 21 of other solvents, "all" = both, or run names separated by commas
MAX_BOXES = 0                   # 0 = no limit; 1 = a timing run of one box
MAX_HOURS = 10                  # start no box that is unlikely to finish in this time (a Colab session ends after at most 12 hours); 0 = no limit
UNITS_PER_HOUR = None           # compute units per hour of this GPU, from the Colab runtime menu; None = no estimate
END_RUNTIME_WHEN_DONE = True    # end the runtime when the run stops, so that no more units are used
CUDA_VERSION = "12.6"           # OpenMM 8.3.1 needs 12.6 or higher (checked with a dry run of the install); the Colab driver must support it

In [ ]:
# The graphics card, and Google Drive.
import subprocess
print(subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total,driver_version",
                      "--format=csv"], capture_output=True, text=True).stdout)
from google.colab import drive
drive.mount("/content/drive")
STORE = f"/content/drive/MyDrive/{DRIVE_FOLDER}/transport"
BUNDLE = f"/content/drive/MyDrive/{DRIVE_FOLDER}/cmf_colab_bundle.zip"

In [ ]:
# Unpack the code from the bundle into the folder /content/cmf.
import os, zipfile, json
assert os.path.isfile(BUNDLE), f"{BUNDLE} is not there: copy cmf_colab_bundle.zip to that folder in Drive"
if os.path.isdir("/content/cmf"):
    import shutil; shutil.rmtree("/content/cmf")
with zipfile.ZipFile(BUNDLE) as z:
    z.extractall("/content/cmf")
print(open("/content/cmf/bundle_info.json").read())

In [ ]:
%%bash -s "$CUDA_VERSION"
# Install the simulation environment with micromamba. The versions are those of the author's
# machine. Several minutes the first time in a session.
set -e
cd /content
if [ ! -x /content/bin/micromamba ]; then
  curl -Ls https://micro.mamba.pm/api/micromamba/linux-64/latest | tar -xj bin/micromamba
fi
export MAMBA_ROOT_PREFIX=/content/mamba
if [ ! -d /content/mamba/envs/md ]; then
  /content/bin/micromamba create -y -q -n md -c conda-forge \
    python=3.11 "cuda-version=$1" openmm=8.3.1 openmmforcefields=0.15.1 \
    openff-toolkit=0.18.0 openff-nagl=0.5.5 openff-nagl-models=2025.9.0 \
    ambertools=24.8 rdkit numpy scipy scikit-learn parmed "pytorch=*=cpu*"
fi
/content/bin/micromamba list -n md | grep -E "^ *(openmm|openmmforcefields|openff-toolkit|ambertools|cuda-version) "

In [ ]:
# Check the environment before any box is run: the CUDA platform of OpenMM, and the tools that
# the benchmark needs. If this fails, the install cell is the place to change (CUDA_VERSION).
import os, subprocess
# Colab sets MPLBACKEND to a backend that only its own Python has; the simulation environment
# would stop at it when it imports matplotlib. "Agg" draws no windows and needs nothing.
ENV = dict(os.environ, MAMBA_ROOT_PREFIX="/content/mamba", PYTHONPATH="/content/cmf",
           PYTHONUNBUFFERED="1", MPLBACKEND="Agg")
MICROMAMBA = ["/content/bin/micromamba", "run", "-n", "md"]
check = """
import openmm
names = [openmm.Platform.getPlatform(i).getName() for i in range(openmm.Platform.getNumPlatforms())]
print("OpenMM", openmm.__version__, "platforms:", names)
system = openmm.System(); system.addParticle(1.0)
context = openmm.Context(system, openmm.VerletIntegrator(0.001), openmm.Platform.getPlatformByName("CUDA"))
print("the CUDA platform works")
from cmfqsar import electrolyte
electrolyte.require()
print("packmol, antechamber and the charge model are available")
"""
out = subprocess.run(MICROMAMBA + ["python", "-c", check], env=ENV, capture_output=True, text=True)
print(out.stdout)
print(out.stderr[-3000:] if out.returncode else "")
assert out.returncode == 0, "the environment check failed: see the text above"

In [ ]:
# Run the boxes. The output of the runner is shown line by line.
import subprocess
cmd = MICROMAMBA + ["python", "/content/cmf/benchmarks/colab/colab_transport.py",
                    "--runs", RUNS, "--store", STORE, "--cache", "/content/cache",
                    "--max-hours", str(MAX_HOURS), "--max-boxes", str(MAX_BOXES)]
proc = subprocess.Popen(cmd, env=ENV, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
for line in proc.stdout:
    if "UFFTYPER" in line or "dcdplugin" in line:
        continue
    print(line, end="")
print("exit code", proc.wait())

In [ ]:
# What was run in this and earlier sessions: time and speed of every box in the store.
import json, os
path = os.path.join(STORE, "colab_log.jsonl")
rows = [json.loads(line) for line in open(path)] if os.path.isfile(path) else []
total = 0.0
for r in rows:
    hours = r["wall_s"] / 3600.0
    total += hours
    print(f'{r["run"]:28s} {r["gpu"]:24s} {hours:5.2f} h {r["ns_per_day"]:6.0f} ns/day {r["atoms"]} atoms')
print(f"{len(rows)} box(es), {total:.2f} hours in all")
if rows and UNITS_PER_HOUR:
    per_box = total / len(rows)
    print(f"about {per_box:.2f} hours and {per_box * UNITS_PER_HOUR:.1f} compute units per box; "
          f"{total * UNITS_PER_HOUR:.1f} units so far")

In [ ]:
# End the runtime so that no more compute units are used.
if END_RUNTIME_WHEN_DONE:
    from google.colab import runtime
    runtime.unassign()